<a href="https://colab.research.google.com/github/GKSJ-AI-CliniScan/MedAssistAI/blob/TahuraShaikh/Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
dataset = pd.read_csv("/content/dataset2_training.csv")

print("Dataset loaded successfully.")
print("Dataset Shape:", dataset.shape)
print("Number of Features:", dataset.shape[1] - 1)
print("Number of Diseases:", dataset["Disease"].nunique())

Dataset loaded successfully.
Dataset Shape: (60000, 378)
Number of Features: 377
Number of Diseases: 658


In [ ]:
X = dataset.drop(columns=["Disease"])
y = dataset["Disease"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)
print("Unique Diseases:", y.nunique())

X Shape: (60000, 377)
y Shape: (60000,)
Unique Diseases: 658


In [ ]:
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

Missing values in X: 0
Missing values in y: 0


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder_rf = LabelEncoder()

y_encoded = label_encoder_rf.fit_transform(y)

print("Number of Classes:", len(label_encoder_rf.classes_))
print("Minimum Label:", y_encoded.min())
print("Maximum Label:", y_encoded.max())

Number of Classes: 658
Minimum Label: 0
Maximum Label: 657


In [ ]:
from sklearn.model_selection import train_test_split

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train_rf.shape)
print("X_test :", X_test_rf.shape)

print("y_train:", y_train_rf.shape)
print("y_test :", y_test_rf.shape)

print("Train classes:", len(np.unique(y_train_rf)))
print("Test classes :", len(np.unique(y_test_rf)))

X_train: (48000, 377)
X_test : (12000, 377)
y_train: (48000,)
y_test : (12000,)
Train classes: 658
Test classes : 614


In [ ]:
print(
    "Missing classes in training:",
    set(range(658)) - set(np.unique(y_train_rf))
)

print(
    "Number of classes missing in testing:",
    len(set(range(658)) - set(np.unique(y_test_rf)))
)

Missing classes in training: set()
Number of classes missing in testing: 44


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Random Forest imported successfully.")

Random Forest imported successfully.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print(rf)

RandomForestClassifier(min_samples_leaf=2, n_estimators=200, n_jobs=-1,
                       random_state=42)


In [ ]:
start_rf = time.time()

rf.fit(
    X_train_rf,
    y_train_rf
)

end_rf = time.time()

rf_training_time = end_rf - start_rf

print(
    "Random Forest Training Time:",
    round(rf_training_time, 2),
    "seconds"
)

Random Forest Training Time: 59.81 seconds


In [ ]:
start_prediction = time.time()

rf_pred = rf.predict(
    X_test_rf
)

end_prediction = time.time()

rf_prediction_time = (
    end_prediction - start_prediction
)

print(
    "Prediction Time:",
    round(rf_prediction_time, 2),
    "seconds"
)

print(
    "Unique Predictions:",
    len(np.unique(rf_pred))
)

Prediction Time: 7.59 seconds
Unique Predictions: 492


In [ ]:
from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(
    y_test_rf,
    rf_pred
)

print(
    "Random Forest Accuracy:",
    round(rf_accuracy * 100, 2),
    "%"
)

Random Forest Accuracy: 80.77 %


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

rf_precision = precision_score(
    y_test_rf,
    rf_pred,
    average="weighted",
    zero_division=0
)

rf_recall = recall_score(
    y_test_rf,
    rf_pred,
    average="weighted",
    zero_division=0
)

rf_f1 = f1_score(
    y_test_rf,
    rf_pred,
    average="weighted",
    zero_division=0
)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("Training Time:", round(rf_training_time, 2), "seconds")

Random Forest Results
---------------------
Accuracy : 0.8077
Precision: 0.8024
Recall   : 0.8077
F1 Score : 0.7985
Training Time: 59.81 seconds


In [ ]:
rf_proba = rf.predict_proba(
    X_test_rf
)

print(
    "Probability Shape:",
    rf_proba.shape
)

print(
    "Number of Model Classes:",
    len(rf.classes_)
)

Probability Shape: (12000, 658)
Number of Model Classes: 658


In [ ]:
from sklearn.metrics import top_k_accuracy_score

top1 = top_k_accuracy_score(
    y_test_rf,
    rf_proba,
    k=1,
    labels=np.arange(658)
)

top3 = top_k_accuracy_score(
    y_test_rf,
    rf_proba,
    k=3,
    labels=np.arange(658)
)

top5 = top_k_accuracy_score(
    y_test_rf,
    rf_proba,
    k=5,
    labels=np.arange(658)
)

print("Random Forest Top-K Accuracy")
print("----------------------------")
print(f"Top-1 Accuracy: {top1 * 100:.2f}%")
print(f"Top-3 Accuracy: {top3 * 100:.2f}%")
print(f"Top-5 Accuracy: {top5 * 100:.2f}%")

Random Forest Top-K Accuracy
----------------------------
Top-1 Accuracy: 80.77%
Top-3 Accuracy: 93.62%
Top-5 Accuracy: 96.72%


In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test_rf,
    rf_pred,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.head()

,precision,recall,f1-score,support
0,1.000000,0.666667,0.800000,3.0
1,0.941176,0.941176,0.941176,17.0
2,0.750000,0.600000,0.666667,10.0
4,1.000000,0.800000,0.888889,10.0
5,0.000000,0.000000,0.000000,1.0


In [ ]:
# Find diseases with the lowest recall
class_recall = report_df.iloc[:-3].sort_values(
    by="recall"
)

print(class_recall.head(20))

     precision  recall  f1-score  support
656        0.0     0.0       0.0      1.0
20         0.0     0.0       0.0      2.0
22         0.0     0.0       0.0      1.0
28         0.0     0.0       0.0      1.0
31         0.0     0.0       0.0      1.0
623        0.0     0.0       0.0      1.0
619        0.0     0.0       0.0      1.0
34         0.0     0.0       0.0      1.0
627        0.0     0.0       0.0      1.0
568        0.0     0.0       0.0      1.0
581        0.0     0.0       0.0      2.0
584        0.0     0.0       0.0      1.0
48         0.0     0.0       0.0      1.0
38         0.0     0.0       0.0      1.0
616        0.0     0.0       0.0      1.0
615        0.0     0.0       0.0      1.0
555        0.0     0.0       0.0      8.0
577        0.0     0.0       0.0      1.0
589        0.0     0.0       0.0      1.0
75         0.0     0.0       0.0      1.0


In [ ]:
# Find diseases with the highest recall
print(class_recall.tail(20))

     precision  recall  f1-score  support
212   0.947368     1.0  0.972973     18.0
213   1.000000     1.0  1.000000      5.0
297   1.000000     1.0  1.000000      2.0
299   1.000000     1.0  1.000000      1.0
302   0.714286     1.0  0.833333     20.0
305   0.818182     1.0  0.900000     18.0
230   1.000000     1.0  1.000000     18.0
278   1.000000     1.0  1.000000      1.0
283   1.000000     1.0  1.000000      2.0
626   1.000000     1.0  1.000000     10.0
599   1.000000     1.0  1.000000      5.0
611   0.900000     1.0  0.947368     18.0
286   1.000000     1.0  1.000000      1.0
240   1.000000     1.0  1.000000      5.0
242   1.000000     1.0  1.000000     10.0
247   1.000000     1.0  1.000000     34.0
253   1.000000     1.0  1.000000      5.0
257   1.000000     1.0  1.000000      3.0
641   0.833333     1.0  0.909091     10.0
642   0.647059     1.0  0.785714     11.0


In [ ]:
# Count samples for each disease in the full dataset
disease_counts = y.value_counts()

print("Disease frequency statistics:")
print(disease_counts.describe())

Disease frequency statistics:
count    658.000000
mean      91.185410
std      109.099227
min        2.000000
25%        8.000000
50%       47.000000
75%      156.000000
max      386.000000
Name: count, dtype: float64


In [ ]:
print("Diseases with 2-10 samples:")
print((disease_counts <= 10).sum())

print("Diseases with 11-50 samples:")
print(((disease_counts > 10) & (disease_counts <= 50)).sum())

print("Diseases with more than 50 samples:")
print((disease_counts > 50).sum())

Diseases with 2-10 samples:
188
Diseases with 11-50 samples:
169
Diseases with more than 50 samples:
301


In [ ]:
class_weight="balanced"

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

start_balanced = time.time()

rf_balanced.fit(
    X_train_rf,
    y_train_rf
)

balanced_training_time = time.time() - start_balanced

print(
    "Balanced Random Forest Training Time:",
    round(balanced_training_time, 2),
    "seconds"
)

In [ ]:

rf_balanced_pred = rf_balanced.predict(X_test_rf)

balanced_accuracy = accuracy_score(
    y_test_rf,
    rf_balanced_pred
)

print(
    "Balanced Random Forest Accuracy:",
    round(balanced_accuracy * 100, 2),
    "%"
)